In [1]:
import requests

url = "https://kamartod.in/"
response = requests.get(url, timeout=20)

print("Status:", response.status_code)
print("Website reachable:", response.status_code == 200)

Status: 200
Website reachable: True


In [2]:
%pip install beautifulsoup4


   ---------------------------------------- 0/2 [soupsieve]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   ---------------------------------------- 2/2 [beautifulsoup4]

Note: you may need to restart the kernel to use updated packages.


In [3]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import requests

url = "https://kamartod.in/"
html = requests.get(url, timeout=20).text
soup = BeautifulSoup(html, "html.parser")

for link in soup.find_all("a", href=True):
    href = urljoin(url, link["href"])
    text = link.get_text(" ", strip=True)

    if any(word in (href + text).lower()
           for word in ["csv", "geojson", "download", "data"]):
        print(text, "->", href)

Open data -> https://kamartod.in/#open-data
Open Database License -> https://opendatacommons.org/licenses/odbl/1-0/
CSV -> https://kamartod.in/api/export.csv
GeoJSON -> https://kamartod.in/api/export.geojson
Data -> https://kamartod.in/api/export.csv


In [4]:
import requests
import pandas as pd
from io import StringIO

url = "https://kamartod.in/api/export.csv"

response = requests.get(url, timeout=30)
response.raise_for_status()

df_road = pd.read_csv(StringIO(response.text))

print("Total records:", len(df_road))
print("Columns:", df_road.columns.tolist())
display(df_road.head())

Total records: 9
Columns: ['id', 'lat', 'lng', 'road_type', 'road_name', 'severity', 'status', 'state', 'district', 'upvotes', 'photo_count', 'first_reported', 'last_reported', 'url']


,id,lat,lng,road_type,road_name,severity,status,state,district,upvotes,photo_count,first_reported,last_reported,url
0,1,12.987959,77.697255,city_road,NaN,2,open,NaN,NaN,8,2,2026-09-27T13:17:49.000Z,2026-09-27T13:18:30.000Z,https://kamartod.in/pothole.html?id=1
1,2,12.944594,77.680272,city_road,NaN,2,open,NaN,NaN,1,1,2026-09-30T12:08:45.000Z,2026-09-30T12:08:45.000Z,https://kamartod.in/pothole.html?id=2
2,3,12.947471,77.680687,city_road,NaN,3,open,NaN,NaN,1,1,2026-09-30T12:10:06.000Z,2026-09-30T12:10:06.000Z,https://kamartod.in/pothole.html?id=3
3,4,12.947533,77.680691,city_road,NaN,2,open,NaN,NaN,1,1,2026-09-30T12:10:38.000Z,2026-09-30T12:10:38.000Z,https://kamartod.in/pothole.html?id=4
4,5,12.948725,77.684794,city_road,NaN,1,open,NaN,NaN,1,1,2026-09-30T12:11:01.000Z,2026-09-30T12:11:01.000Z,https://kamartod.in/pothole.html?id=5


In [5]:
# Keep records with valid coordinates
df_road = df_road.dropna(subset=["lat", "lng"]).copy()

# Convert numeric columns
for col in ["lat", "lng", "severity", "upvotes", "photo_count"]:
    df_road[col] = pd.to_numeric(df_road[col], errors="coerce")

# Remove invalid coordinates
df_road = df_road[
    df_road["lat"].between(-90, 90) &
    df_road["lng"].between(-180, 180)
].copy()

# Create a basic road safety score
# Higher severity means lower safety
df_road["road_score"] = (
    100 * (1 - df_road["severity"].fillna(0) / 3)
).clip(0, 100).round(2)

print("Clean records:", len(df_road))
display(df_road[[
    "lat", "lng", "severity", "status", "road_score"
]])

Clean records: 9


,lat,lng,severity,status,road_score
0,12.987959,77.697255,2,open,33.33
1,12.944594,77.680272,2,open,33.33
2,12.947471,77.680687,3,open,0.00
3,12.947533,77.680691,2,open,33.33
4,12.948725,77.684794,1,open,66.67
5,12.954136,77.689282,3,open,0.00
6,12.972960,77.693584,2,open,33.33
7,12.972989,77.694038,3,open,0.00
8,12.980571,77.694989,2,open,33.33


In [6]:
import os

os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/road_condition_scores.csv"

df_road.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Total records:", len(df_road))

Saved to: ../data/processed/road_condition_scores.csv
Total records: 9


In [7]:
print("States:", df_road["state"].dropna().unique())
print("Districts:", df_road["district"].dropna().unique())

print("\nReport status:")
print(df_road["status"].value_counts(dropna=False))

print("\nCoordinates range:")
print("Latitude:", df_road["lat"].min(), "to", df_road["lat"].max())
print("Longitude:", df_road["lng"].min(), "to", df_road["lng"].max())

States: []
Districts: []

Report status:
status
open    9
Name: count, dtype: int64

Coordinates range:
Latitude: 12.944594 to 12.987959
Longitude: 77.680272 to 77.697255


In [8]:
import requests

url = "https://kamartod.in/api/export.geojson"
response = requests.get(url, timeout=30)

print("Status:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print(response.text[:500])

Status: 200
Content type: application/geo+json; charset=utf-8
{"type":"FeatureCollection","license":"Open Database License (ODbL) 1.0, https://opendatacommons.org/licenses/odbl/1-0/","generated":"2026-10-24T05:52:39.846Z","features":[
{"type":"Feature","geometry":{"type":"Point","coordinates":[77.697255,12.987959]},"properties":{"id":1,"road_type":"city_road","road_name":"","severity":2,"status":"open","state":"","district":"","upvotes":8,"photo_count":2,"first_reported":"2026-09-27T13:17:49.000Z","last_reported":"2026-09-27T13:18:30.000Z","url":"https://k


In [9]:
geojson_data = response.json()

features = geojson_data.get("features", [])

print("Total GeoJSON reports:", len(features))
print("Generated at:", geojson_data.get("generated"))
print("License:", geojson_data.get("license"))

Total GeoJSON reports: 9
Generated at: 2026-10-24T05:52:39.846Z
License: Open Database License (ODbL) 1.0, https://opendatacommons.org/licenses/odbl/1-0/


In [10]:
import os
import shutil

os.makedirs("../data/final", exist_ok=True)

shutil.copy2(
    "../data/processed/road_condition_scores.csv",
    "../data/final/road_condition_scores.csv"
)

print("Road dataset copied to data/final!")

Road dataset copied to data/final!


In [1]:

import pandas as pd
from pathlib import Path

# Load the original road condition dataset
input_path = "../data/final/road_condition_scores.csv"
df_road = pd.read_csv(input_path)

# Select fields needed for backend integration
backend_columns = [
    "id",
    "lat",
    "lng",
    "road_type",
    "road_name",
    "severity",
    "status",
    "state",
    "district",
    "road_score",
    "first_reported",
    "last_reported",
    "url"
]

df_backend = df_road[backend_columns].copy()

# Replace missing descriptive fields without inventing data
text_columns = [
    "road_name",
    "state",
    "district",
    "first_reported",
    "last_reported",
    "url"
]

for column in text_columns:
    df_backend[column] = df_backend[column].fillna("Unknown")

# Rename coordinates for consistency with other datasets
df_backend = df_backend.rename(
    columns={
        "lat": "latitude",
        "lng": "longitude"
    }
)

# Save as a separate backend file
output_path = Path("../data/final/road_condition_backend.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df_backend.to_csv(output_path, index=False)

# Verify the output
df_check = pd.read_csv(output_path)

print("Saved file:", output_path)
print("Total records:", len(df_check))
print("Missing values:", int(df_check.isna().sum().sum()))
print("Columns:", df_check.columns.tolist())

display(df_check.head(9))


Saved file: ..\data\final\road_condition_backend.csv
Total records: 9
Missing values: 0
Columns: ['id', 'latitude', 'longitude', 'road_type', 'road_name', 'severity', 'status', 'state', 'district', 'road_score', 'first_reported', 'last_reported', 'url']


,id,latitude,longitude,road_type,road_name,severity,status,state,district,road_score,first_reported,last_reported,url
0,1,12.987959,77.697255,city_road,Unknown,2,open,Unknown,Unknown,33.33,2026-09-27T13:17:49.000Z,2026-09-27T13:18:30.000Z,https://kamartod.in/pothole.html?id=1
1,2,12.944594,77.680272,city_road,Unknown,2,open,Unknown,Unknown,33.33,2026-09-30T12:08:45.000Z,2026-09-30T12:08:45.000Z,https://kamartod.in/pothole.html?id=2
2,3,12.947471,77.680687,city_road,Unknown,3,open,Unknown,Unknown,0.00,2026-09-30T12:10:06.000Z,2026-09-30T12:10:06.000Z,https://kamartod.in/pothole.html?id=3
3,4,12.947533,77.680691,city_road,Unknown,2,open,Unknown,Unknown,33.33,2026-09-30T12:10:38.000Z,2026-09-30T12:10:38.000Z,https://kamartod.in/pothole.html?id=4
4,5,12.948725,77.684794,city_road,Unknown,1,open,Unknown,Unknown,66.67,2026-09-30T12:11:01.000Z,2026-09-30T12:11:01.000Z,https://kamartod.in/pothole.html?id=5
5,6,12.954136,77.689282,city_road,Unknown,3,open,Unknown,Unknown,0.00,2026-09-30T12:17:46.000Z,2026-09-30T12:17:46.000Z,https://kamartod.in/pothole.html?id=6
6,7,12.972960,77.693584,local_street,Unknown,2,open,Unknown,Unknown,33.33,2026-09-30T12:38:22.000Z,2026-09-30T12:38:22.000Z,https://kamartod.in/pothole.html?id=7
7,8,12.972989,77.694038,local_street,Unknown,3,open,Unknown,Unknown,0.00,2026-09-30T12:41:34.000Z,2026-09-30T12:41:34.000Z,https://kamartod.in/pothole.html?id=8
8,9,12.980571,77.694989,city_road,Unknown,2,open,Unknown,Unknown,33.33,2026-09-30T13:00:27.000Z,2026-09-30T13:00:27.000Z,https://kamartod.in/pothole.html?id=9
